In [1]:
from pathlib import Path
import sys

cwd = Path.cwd()
if (cwd / "src").exists():
    project_root = cwd
elif (cwd.parent / "src").exists():
    project_root = cwd.parent
else:
    raise FileNotFoundError("Could not locate project root containing 'src'.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


In [12]:
import numpy as np
import pandas as pd
from IPython.display import display
import statsmodels.api as sm


In [13]:

from src.data.create_dataframes import processed_df_map

pd.set_option("display.float_format", "{:,.4f}".format)

DEFAULT_Y_COL_MAP = {
    "eur_df_processed": "EURUSD",
    "gbp_df_processed": "GBPUSD",
    "jpy_df_processed": "USDJPY",
    "chf_df_processed": "USDCHF",
    "cad_df_processed": "USDCAD",
    "aud_df_processed": "AUDUSD",
    "nzd_df_processed": "NZDUSD",
    "nok_df_processed": "USDNOK",
    "sek_df_processed": "USDSEK",
}

def latest_window_multivariate_ols(df: pd.DataFrame, y_col: str, window: int = 252):
    """Fit multivariate OLS on the most recent rolling window and return (summary, r2)."""
    if y_col not in df.columns:
        raise KeyError(f"{y_col} not found in dataframe columns.")

    window_df = df.tail(window).dropna()
    if len(window_df) < 2:
        raise ValueError("Not enough non-NaN rows in the latest rolling window.")

    y = window_df[y_col]
    X = window_df.drop(columns=[y_col])
    X_const = sm.add_constant(X, has_constant="add")

    model = sm.OLS(y, X_const).fit()
    r2 = model.rsquared

    betas = model.params.drop("const")
    pvals = model.pvalues.drop("const")
    signif_pct = (1 - pvals) * 100

    ranked_idx = betas.abs().sort_values(ascending=False).index
    betas = betas.reindex(ranked_idx)
    signif_pct = signif_pct.reindex(ranked_idx)

    summary = pd.DataFrame({
        "Driver Name": betas.index,
        "Beta Coefficient": betas.values,
        "Rank": np.arange(1, len(betas) + 1),
        "Significance %": signif_pct.values,
    })
    summary["Beta Coefficient"] = summary["Beta Coefficient"].round(4)
    summary["Significance %"] = summary["Significance %"].round(2)
    return summary, r2

def run_processed_df_regression(df_name: str, y_col: str | None = None, window: int = 252):
    """Convenience wrapper for processed_df_map by name."""
    if df_name not in processed_df_map:
        raise KeyError(f"{df_name} not found in processed_df_map.")

    if y_col is None:
        y_col = DEFAULT_Y_COL_MAP.get(df_name)
        if y_col is None:
            raise KeyError(f"No default y_col found for {df_name}.")

    summary, r2 = latest_window_multivariate_ols(processed_df_map[df_name], y_col, window=window)
    display(summary)
    print(f"Current 1Y Rolling R2: {r2:.4f}")
    return summary, r2




In [3]:
sorted(processed_df_map.keys())


['aud_df_processed',
 'cad_df_processed',
 'chf_df_processed',
 'eur_df_processed',
 'gbp_df_processed',
 'jpy_df_processed',
 'nok_df_processed',
 'nzd_df_processed',
 'sek_df_processed']

In [4]:
# Example usage (default y_col is picked automatically)
run_processed_df_regression("nok_df_processed")


,Driver Name,Beta Coefficient,Rank,Significance %
0,5y yield,3.5072,1,85.8100
1,10y yield,-2.0150,2,74.5500
2,BBDXY,0.8564,3,100.0000
3,EMFX,-0.5941,4,99.9200
4,Real 2y yield,-0.3468,5,72.8300
5,MSCI World - S&P500,0.1353,6,30.1800
6,BCOM Index,-0.1199,7,96.6500
7,Local - S&P500,0.0999,8,41.6600
8,2y yield,0.0956,9,5.9000
9,Local Index,-0.0786,10,33.1400


Current 1Y Rolling R2: 0.7317


(               Driver Name  Beta Coefficient  Rank  Significance %
 0                 5y yield            3.5072     1         85.8100
 1                10y yield           -2.0150     2         74.5500
 2                    BBDXY            0.8564     3        100.0000
 3                     EMFX           -0.5941     4         99.9200
 4            Real 2y yield           -0.3468     5         72.8300
 5      MSCI World - S&P500            0.1353     6         30.1800
 6               BCOM Index           -0.1199     7         96.6500
 7           Local - S&P500            0.0999     8         41.6600
 8                 2y yield            0.0956     9          5.9000
 9              Local Index           -0.0786    10         33.1400
 10                     Oil           -0.0677    11         99.9800
 11        Local - Wilshire           -0.0482    12         19.3200
 12         TTF Natural Gas           -0.0438    13         79.1500
 13          UK Natural Gas            0.0372   

In [5]:
def run_processed_df_regression_sig_only(
    df_name: str,
    y_col: str | None = None,
    window: int = 252,
    min_significance: float = 95.0,
):
    """Same as run_processed_df_regression but filters to >= min_significance."""
    summary, r2 = run_processed_df_regression(df_name, y_col=y_col, window=window)
    filtered = summary[summary["Significance %"] >= min_significance].reset_index(drop=True)
    display(filtered)
    print(f"Current 1Y Rolling R2: {r2:.4f}")
    return filtered, r2

run_processed_df_regression_sig_only("aud_df_processed")



,Driver Name,Beta Coefficient,Rank,Significance %
0,2y yield,3.6053,1,98.3400
1,5y yield,-3.2437,2,85.9200
2,BBDXY,-1.0952,3,100.0000
3,MSCI World - Wilshire,0.4209,4,93.5200
4,Local Index,0.3719,5,97.9600
5,Local - S&P500,-0.3420,6,96.2600
6,MSCI World - Dow Jones,0.2041,7,93.1900
7,Real 2y yield,-0.1993,8,51.2800
8,MSCI World ex-US,-0.1674,9,71.4500
9,10y yield,0.1343,10,8.5600


Current 1Y Rolling R2: 0.7728


,Driver Name,Beta Coefficient,Rank,Significance %
0,2y yield,3.6053,1,98.3400
1,BBDXY,-1.0952,3,100.0000
2,Local Index,0.3719,5,97.9600
3,Local - S&P500,-0.3420,6,96.2600
4,Coking coal,-0.0319,16,96.8300
5,G10 FX Volatility,-0.0142,17,96.2500


Current 1Y Rolling R2: 0.7728


(         Driver Name  Beta Coefficient  Rank  Significance %
 0           2y yield            3.6053     1         98.3400
 1              BBDXY           -1.0952     3        100.0000
 2        Local Index            0.3719     5         97.9600
 3     Local - S&P500           -0.3420     6         96.2600
 4        Coking coal           -0.0319    16         96.8300
 5  G10 FX Volatility           -0.0142    17         96.2500,
 np.float64(0.7728044417673012))

In [6]:
from src.data.create_dataframes import build_df2_map, frames, fx_newdf

# Assume you have frames and fx_newdf already loaded
df2_map = build_df2_map(frames, fx_newdf)

# To get the dataframe:
nok_df2 = df2_map["nok_df2"]
aud_df2 = df2_map["aud_df2"]
eur_df2 = df2_map["eur_df2"]


In [7]:
aud_df2.tail()

,GTAUD2YR Corp,GTAUD5YR Corp,GTAUD10YR Corp,RR2YHAU Index,AS51 Index,SCOH6 COMB Comdty,IACA COMB Comdty,USGG2YR Index,USGG5YR Index,USGG10YR Index,...,TZT1 Comdty,NG1 COMB Comdty,HGA Comdty,LMCADS03 Comdty,VIX Index,MOVE Index,JPMVG71M Index,AUDUSD Curncy,BBDXY_ex_AUD,CTTWBUSA Index
2026-02-13,4.2410,4.3880,4.7480,NaN,"8,917.6120",96.7500,222.0000,3.4076,3.6038,4.0483,...,32.5000,3.2430,586.2500,"12,881.0000",20.6000,70.1000,7.7300,0.7073,"1,218.2994",103.3611
2026-02-16,4.2220,4.3620,4.7090,NaN,"8,937.0950",96.6500,222.0000,3.4076,3.6038,4.0483,...,30.9030,NaN,NaN,"12,850.5000",21.2000,NaN,7.3100,0.7072,"1,219.7648",103.1725
2026-02-17,4.1860,4.3310,4.6850,NaN,"8,958.8780",96.1900,221.0000,3.4325,3.6210,4.0578,...,29.8220,3.0310,570.4000,"12,619.5000",20.2900,68.8400,7.3000,0.7086,"1,219.8220",103.2271
2026-02-18,4.2190,4.3680,4.7230,NaN,"9,006.9840",95.7200,221.0000,3.4637,3.6504,4.0769,...,31.4100,2.9740,584.8000,NaN,18.9800,NaN,7.3000,0.7069,"1,222.6896",103.2615
NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
from src.data.build_ultimate_df import build_ultimate_df

ultimate_df = build_ultimate_df()
ultimate_df["aud"].tail()


,AUDUSD,BBDXY,Asia EMFX,2y yield,5y yield,10y yield,Real 2y yield,Local - S&P500,Local - Wilshire,Local - Dow Jones,...,Copper COMEX,Copper LME,MOVE Index,JPMVG71M Index,NG1 COMB Comdty,TZT1 Comdty,FN1 Comdty,VIX Index,SCOH6 COMB Comdty,IACA COMB Comdty
2026-02-13,-0.2401,-0.0285,0.2090,0.0083,0.0087,-0.0092,0.0483,-1.4522,-1.5735,-1.5012,...,0.2904,0.0427,70.1000,7.7300,3.2430,32.5000,76.2500,20.6000,96.7500,222.0000
2026-02-16,-0.0141,0.1202,-0.1826,-0.0190,-0.0260,-0.0390,0.0000,0.2182,0.2182,0.2182,...,0.0000,-0.2371,NaN,7.3100,NaN,30.9030,72.5500,21.2000,96.6500,222.0000
2026-02-17,0.1978,0.0047,0.0529,-0.0609,-0.0482,-0.0335,-0.0249,0.1404,0.1515,0.1783,...,-2.7408,-1.8139,68.8400,7.3000,3.0310,29.8220,71.1300,20.2900,96.1900,221.0000
2026-02-18,-0.2402,0.2348,0.0333,0.0018,0.0076,0.0189,-0.0312,-0.2073,-0.2919,-0.0833,...,2.4932,0.0000,NaN,7.3000,2.9740,31.4100,76.4100,18.9800,95.7200,221.0000
NaT,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
nok_ultimate_df = ultimate_df["nok"]
nok_ultimate_df

,USDNOK,BBDXY,EMFX,2y yield,5y yield,10y yield,Real 2y yield,Local - S&P500,Local - Wilshire,Local - Dow Jones,...,Gold,Oil,Copper COMEX,Copper LME,MOVE Index,JPMVG71M Index,NG1 COMB Comdty,TZT1 Comdty,FN1 Comdty,VIX Index
2000-03-02,0.7499,NaN,-0.1204,NaN,NaN,NaN,0.0010,-0.1590,0.0945,-0.2387,...,-0.4323,0.5491,NaN,-1.1915,99.6800,NaN,2.7830,NaN,11.7200,21.0600
2000-03-03,0.7502,NaN,-0.0803,NaN,NaN,NaN,0.0180,-1.2876,-1.6072,-1.2938,...,0.1039,-0.7902,NaN,0.5124,94.8500,NaN,2.8250,NaN,11.6500,19.2100
2000-03-06,0.1211,NaN,-0.0502,NaN,NaN,NaN,-0.0240,1.9164,1.4917,2.5542,...,-0.4059,2.1836,NaN,0.3967,92.0600,NaN,2.8500,NaN,11.6700,21.5000
2000-03-07,-0.1045,NaN,0.0301,NaN,NaN,NaN,0.0130,5.6277,5.2075,6.7826,...,1.9351,7.3819,NaN,-1.0805,97.1300,NaN,2.7990,NaN,11.6900,24.3100
2000-03-08,0.0048,NaN,0.1105,NaN,NaN,NaN,-0.0190,-1.8061,-1.6454,-1.6078,...,-1.7614,-10.2231,NaN,-1.2080,103.6700,NaN,2.7100,NaN,11.8500,23.8200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-13,-0.2583,-0.0685,-0.0281,-0.0203,-0.0107,-0.0088,0.0483,-0.9480,-1.0693,-0.9970,...,2.4057,0.3401,0.2904,0.0427,70.1000,7.7300,3.2430,32.5000,76.2500,20.6000
2026-02-16,-0.0252,0.1353,0.0000,0.0170,0.0080,0.0090,0.0000,0.0935,0.0935,0.0935,...,-0.9958,1.3197,0.0000,-0.2371,NaN,7.3100,NaN,30.9030,72.5500,21.2000
2026-02-17,0.2584,0.0296,0.0000,0.0939,0.0672,0.0635,-0.0249,-0.5064,-0.4952,-0.4684,...,-2.3140,-1.8079,-2.7408,-1.8139,68.8400,7.3000,3.0310,29.8220,71.1300,20.2900
2026-02-18,-0.4500,0.2085,0.0000,0.0362,0.0284,0.0021,-0.0312,1.4279,1.3433,1.5519,...,2.4331,2.9953,2.4932,0.0000,NaN,7.3000,2.9740,31.4100,76.4100,18.9800


In [10]:
sek_ultimate_df = ultimate_df["sek"]
sek_ultimate_df

,USDSEK,BBDXY,EMFX,2y yield,5y yield,10y yield,Real 2y yield,Local - S&P500,Local - Wilshire,Local - Dow Jones,...,Gold,Oil,Copper COMEX,Copper LME,MOVE Index,JPMVG71M Index,NG1 COMB Comdty,TZT1 Comdty,FN1 Comdty,VIX Index
2000-03-02,1.3433,NaN,-0.1204,NaN,NaN,NaN,0.0210,3.2961,3.5496,3.2164,...,-0.4323,0.5491,NaN,-1.1915,99.6800,NaN,2.7830,NaN,11.7200,21.0600
2000-03-03,0.2549,NaN,-0.0803,NaN,NaN,NaN,0.0080,-1.0187,-1.3382,-1.0248,...,0.1039,-0.7902,NaN,0.5124,94.8500,NaN,2.8250,NaN,11.6500,19.2100
2000-03-06,-0.0853,NaN,-0.0502,NaN,NaN,NaN,-0.0240,1.7543,1.3296,2.3921,...,-0.4059,2.1836,NaN,0.3967,92.0600,NaN,2.8500,NaN,11.6700,21.5000
2000-03-07,-0.1491,NaN,0.0301,NaN,NaN,NaN,0.0430,2.7051,2.2848,3.8600,...,1.9351,7.3819,NaN,-1.0805,97.1300,NaN,2.7990,NaN,11.6900,24.3100
2000-03-08,0.3843,NaN,0.1105,NaN,NaN,NaN,-0.0290,-4.6704,-4.5097,-4.4721,...,-1.7614,-10.2231,NaN,-1.2080,103.6700,NaN,2.7100,NaN,11.8500,23.8200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-13,-0.0639,-0.0685,-0.0281,-0.0213,-0.0177,-0.0138,0.0483,-0.1746,-0.2959,-0.2236,...,2.4057,0.3401,0.2904,0.0427,70.1000,7.7300,3.2430,32.5000,76.2500,20.6000
2026-02-16,0.2787,0.1353,0.0000,0.0040,0.0030,0.0010,0.0000,0.0656,0.0656,0.0656,...,-0.9958,1.3197,0.0000,-0.2371,NaN,7.3100,NaN,30.9030,72.5500,21.2000
2026-02-17,0.2344,0.0296,0.0000,0.0319,0.0262,0.0155,-0.0249,0.1631,0.1742,0.2010,...,-2.3140,-1.8079,-2.7408,-1.8139,68.8400,7.3000,3.0310,29.8220,71.1300,20.2900
2026-02-18,0.2595,0.2085,0.0000,0.0352,0.0284,0.0151,-0.0312,0.4627,0.3781,0.5867,...,2.4331,2.9953,2.4932,0.0000,NaN,7.3000,2.9740,31.4100,76.4100,18.9800


In [14]:
from src.data.build_ultimate_df import build_ultimate_df

ultimate_df = build_ultimate_df()

pd.set_option("display.float_format", "{:,.4f}".format)

DEFAULT_Y_COL_MAP = {
    "eur": "EURUSD",
    "gbp": "GBPUSD",
    "jpy": "USDJPY",
    "chf": "USDCHF",
    "cad": "USDCAD",
    "aud": "AUDUSD",
    "nzd": "NZDUSD",
    "nok": "USDNOK",
    "sek": "USDSEK",
}

def latest_window_multivariate_ols(df: pd.DataFrame, y_col: str, window: int = 252):
    """Fit multivariate OLS on the most recent rolling window and return (summary, r2)."""
    if y_col not in df.columns:
        raise KeyError(f"{y_col} not found in dataframe columns.")

    window_df = df.tail(window).dropna()
    if len(window_df) < 2:
        raise ValueError("Not enough non-NaN rows in the latest rolling window.")

    y = window_df[y_col]
    X = window_df.drop(columns=[y_col])
    X_const = sm.add_constant(X, has_constant="add")

    model = sm.OLS(y, X_const).fit()
    r2 = model.rsquared

    betas = model.params.drop("const")
    pvals = model.pvalues.drop("const")
    signif_pct = (1 - pvals) * 100

    ranked_idx = betas.abs().sort_values(ascending=False).index
    betas = betas.reindex(ranked_idx)
    signif_pct = signif_pct.reindex(ranked_idx)

    summary = pd.DataFrame({
        "Driver Name": betas.index,
        "Beta Coefficient": betas.values,
        "Rank": np.arange(1, len(betas) + 1),
        "Significance %": signif_pct.values,
    })
    summary["Beta Coefficient"] = summary["Beta Coefficient"].round(4)
    summary["Significance %"] = summary["Significance %"].round(2)
    return summary, r2

def run_processed_df_regression(df_name: str, y_col: str | None = None, window: int = 252):
    """Convenience wrapper for ultimate_df by name."""
    if df_name not in ultimate_df:
        raise KeyError(f"{df_name} not found in ultimate_df.")

    if y_col is None:
        y_col = DEFAULT_Y_COL_MAP.get(df_name)
        if y_col is None:
            raise KeyError(f"No default y_col found for {df_name}.")

    summary, r2 = latest_window_multivariate_ols(ultimate_df[df_name], y_col, window=window)
    display(summary)
    print(f"Current 1Y Rolling R2: {r2:.4f}")
    return summary, r2




In [15]:
def run_processed_df_regression_sig_only(
    df_name: str,
    y_col: str | None = None,
    window: int = 252,
    min_significance: float = 95.0,
):
    """Same as run_processed_df_regression but filters to >= min_significance."""
    summary, r2 = run_processed_df_regression(df_name, y_col=y_col, window=window)
    filtered = summary[summary["Significance %"] >= min_significance].reset_index(drop=True)
    display(filtered)
    print(f"Current 1Y Rolling R2: {r2:.4f}")
    return filtered, r2

run_processed_df_regression_sig_only("aud")


,Driver Name,Beta Coefficient,Rank,Significance %
0,2y yield,3.1133,1,94.4600
1,5y yield,-2.2681,2,67.0900
2,BBDXY,-1.0314,3,100.0000
3,10y yield,-0.5325,4,32.0800
4,Local Index,0.3870,5,70.4700
5,MSCI World - Wilshire,0.3295,6,55.6400
6,Local - S&P500,-0.3169,7,91.3600
7,MSCI World - Dow Jones,0.1955,8,53.2400
8,Real 2y yield,-0.1519,9,37.7400
9,MSCI World ex-US,-0.1066,10,22.3700


Current 1Y Rolling R2: 0.7754


,Driver Name,Beta Coefficient,Rank,Significance %
0,BBDXY,-1.0314,3,100.0000


Current 1Y Rolling R2: 0.7754


(  Driver Name  Beta Coefficient  Rank  Significance %
 0       BBDXY           -1.0314     3        100.0000,
 np.float64(0.7754355093250985))